In [ ]:
# читаешь ссылки, создаешь класс для датасета

In [ ]:
from collections import Counter

#  x - ссылки на картинки которые получишь выше

data_dict = dict(Counter([x.parent.name for x in train_val_files]))
data = pd.DataFrame(data = data_dict.values(), index=data_dict.keys(), columns=['count'])
plt.figure(figsize=(20,10))
sns.barplot(x = data.index, y = data['count']).set_xticklabels(data.index, rotation=90)
plt.show()

In [ ]:
# посмотрев на дисбаланс фиксишь значение is_enght, ставишь крч сколько считаешь нужным
is_enght = data['count'] < 1500
data.loc[is_enght, 'add'] = (1500 - data['count']).astype(int)
data.loc[~is_enght, 'add'] = 0
data['from_one_image'] = (np.ceil(data['add'] / data['count'])).astype(int)
data

In [ ]:
# твои аугментации:
augmenters = {}

In [ ]:
# тест аугментаций 

train_dataset = Picture_Dataset(train_val_files, mode='train')

fig, ax = plt.subplots(nrows=5, ncols=(len(augmenters) + 1),figsize=(10, 10))

for i in range(5):
    random_class = int(np.random.uniform(0, len(train_val_files)))
    img_orig = train_dataset.load_sample(train_val_files[random_class])
    img_label = train_val_files[random_class].parent.name
    
    ax[i][0].imshow(img_orig)
    ax[i][0].set_title(img_label)
    ax[i][0].axis('off')
        
    for j, (augmenter_name, augmenter) in enumerate(augmenters.items()):
        img_aug = augmenter(img_orig)
        ax[i][j + 1].imshow(img_aug)
        ax[i][j + 1].set_title(augmenter_name)
        ax[i][j + 1].axis('off')

In [ ]:
# самое интересное
# ссылки само собой ставишь свои

import os

create_dir = Path('/content/drive/MyDrive/sample')

if not os.path.isdir(create_dir):
    os.mkdir(create_dir)

proc_dataset = Picture_Dataset(train_files, mode='train') # тут используй любой свой класс для входа датасета

for image_path in tqdm(train_files):
    path = image_path.parents[1]
    x_class = image_path.parent.parent.name
    img = proc_dataset.load_sample(image_path)
    
    # все эти условия ниже нужны для фикса дисбаланса, если просто семплишь то удаляй всё что до цикла и создание датафрейма для фикса
    if data.loc[x_class]['add'] <= 0:
        continue
  
    if data.loc[x_class]['from_one_image'] > data.loc[x_class]['add']:
        iter_size = data.loc[x_class]['add']
    else:
        iter_size = data.loc[x_class]['from_one_image']
    data.loc[x_class]['add'] -= iter_size
    
    for i in range(int(iter_size)):
        
        parent_dir = Path('/content/drive/MyDrive/sample')
        
        directory = x_class
        
        path = os.path.join(parent_dir, directory, 'image')
        
        if not os.path.isdir(path):
            os.mkdir(path)
        
        augmenter = random.choice(list(augmenters.values()))
        aug_img = augmenter(img)
        aug_img.save(f"{path}/{image_path.name.split('.')[0]}_{i}.jpg")

In [ ]:
# после не забудь обновить датасет ссылок если он уже создан